# Animations for the KSIAM talk

This notebook writes the talk's animations, as GIF and MP4, and their static strips to `results/ksiam/`. The outputs are not committed.

## Setup

The style, the ffmpeg path and the panel-2 system.

In [1]:
import copy
from pathlib import Path

import imageio_ffmpeg
import jax
import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from deep_isochron.plotting import (
    animate_isochron_sync,
    animate_kick,
    configure_rcparams,
    PortraitStyle,
)
from deep_isochron.systems.normal_forms import BautinNormalForm
from matplotlib.animation import FFMpegWriter, PillowWriter
from PIL import Image


jax.config.update("jax_enable_x64", True)
configure_rcparams()
mpl.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()

style = PortraitStyle()
# a = 1/3, b = 0.5 and w0 = w - 2/3 with w = 2 pi give the period T = 1 and mu = 0.37
system = BautinNormalForm(a=1 / 3, b=0.5, w=2 * jnp.pi, w0=2 * jnp.pi - 2 / 3)
out = Path("results/ksiam")
out.mkdir(parents=True, exist_ok=True)

## A1: a kick shifts the phase

A point on the cycle and two kicked copies, one delayed and one advanced, in `a1_kick.gif` and `a1_kick.mp4`.

In [2]:
length = 0.75
kicks = jnp.array([[length, 0.0], [-length, 0.0]])
a1 = animate_kick(system, kick_phase=0.83, kicks=kicks, n_periods=4, style=style)
a1.save(out / "a1_kick.gif", writer=PillowWriter(fps=style.fps), dpi=style.dpi)
a1.save(
    out / "a1_kick.mp4",
    writer=FFMpegWriter(
        fps=style.fps, codec="libx264", extra_args=["-pix_fmt", "yuv420p"]
    ),
    dpi=style.dpi,
)
plt.close(a1._fig)

## A2: the points of an isochron fall into step

Points seeded on the isochrons of phases 0 and 1/2, without tails, in `a2_sync.gif` and `a2_sync.mp4`.

In [3]:
a2_style = copy.replace(style, tail_periods=0)
a2 = animate_isochron_sync(
    system,
    phases=jnp.array([0.0, 0.5]),
    radii=jnp.linspace(0.5, 1.6, 5),
    n_periods=5,
    style=a2_style,
)
a2.save(out / "a2_sync.gif", writer=PillowWriter(fps=style.fps), dpi=style.dpi)
a2.save(
    out / "a2_sync.mp4",
    writer=FFMpegWriter(
        fps=style.fps, codec="libx264", extra_args=["-pix_fmt", "yuv420p"]
    ),
    dpi=style.dpi,
)
plt.close(a2._fig)

## Strips

Stills from the saved GIFs, side by side. Pillow merges identical consecutive frames and the pauses repeat frames, so a frame is located by its cumulative display time, not by its index.

In [4]:
def display_seconds(t: float, pause_frame: int) -> float:
    """The display time, in seconds, in the middle of the frame at scene time `t` in
    periods; every frame after `pause_frame` is shifted by the pause."""
    j = round(t * style.frames_per_period)
    pause = style.pause_seconds if j > pause_frame else 0.0
    return (j + 0.5) / style.fps + pause


def gif_frame(path: Path, seconds: float) -> np.ndarray:
    """The frame of the GIF at `path` that is on display at `seconds`."""
    with Image.open(path) as gif:
        elapsed = 0.0
        for index in range(gif.n_frames):
            gif.seek(index)
            elapsed += gif.info["duration"] / 1000
            if seconds < elapsed:
                return np.asarray(gif.convert("RGB"))
        raise ValueError(f"{seconds} s is past the end of {path}, at {elapsed} s.")


def gif_duration(path: Path) -> float:
    """The total display time of the GIF at `path`, in seconds."""
    with Image.open(path) as gif:
        total = 0.0
        for index in range(gif.n_frames):
            gif.seek(index)
            total += gif.info["duration"] / 1000
    return total


def save_strip(frames: list[np.ndarray], path: Path) -> None:
    fig, axes = plt.subplots(1, len(frames), figsize=(6 * len(frames), 2.5))
    for ax, frame in zip(axes, frames):
        ax.imshow(frame)
        ax.axis("off")
    fig.savefig(path, dpi=style.dpi, bbox_inches="tight")
    plt.close(fig)


a1_gif, a2_gif = out / "a1_kick.gif", out / "a2_sync.gif"
kick_frame = style.frames_per_period
save_strip(
    [
        gif_frame(a1_gif, display_seconds(0, kick_frame)),
        gif_frame(a1_gif, display_seconds(1, kick_frame)),
        gif_frame(a1_gif, gif_duration(a1_gif) - 0.5 / style.fps),
    ],
    out / "a1_strip.png",
)
save_strip(
    [gif_frame(a2_gif, display_seconds(t, 0)) for t in range(4)],
    out / "a2_strip.png",
)